# Hassan Project 3 — Customer Churn Prediction & Retention Analytics

**Python + Machine Learning + Power BI**

This notebook is designed to run **top-to-bottom in Google Colab**.

### What this notebook does
- Downloads the public IBM Telco Customer Churn dataset automatically
- Cleans and validates the data
- Performs exploratory churn analysis
- Engineers business-friendly features
- Trains and compares multiple classification models
- Evaluates Accuracy, Precision, Recall, F1 and ROC-AUC
- Builds churn-risk scores for customers
- Exports Power BI-ready data
- Saves charts, model files and business findings
- Creates a ZIP containing the project outputs

> Run cells in order using **Runtime → Run all**.

## 1. Imports and project folders

The notebook uses standard Python data-science libraries available in Colab.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)
from sklearn.inspection import permutation_importance
import joblib

PROJECT_DIR = Path("project_3_outputs")
DATA_DIR = PROJECT_DIR / "data"
IMAGES_DIR = PROJECT_DIR / "images"
MODELS_DIR = PROJECT_DIR / "models"
REPORTS_DIR = PROJECT_DIR / "reports"
POWERBI_DIR = PROJECT_DIR / "powerbi"
SRC_DIR = PROJECT_DIR / "src"

for folder in [DATA_DIR, IMAGES_DIR, MODELS_DIR, REPORTS_DIR, POWERBI_DIR, SRC_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folders created:")
for folder in [DATA_DIR, IMAGES_DIR, MODELS_DIR, REPORTS_DIR, POWERBI_DIR, SRC_DIR]:
    print("-", folder)

## 2. Download the IBM Telco Customer Churn dataset

The notebook tries more than one public source so it is more robust if one URL becomes unavailable.

In [ ]:
import urllib.request

DATA_PATH = DATA_DIR / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

DATA_URLS = [
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv",
    "https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv",
    "https://raw.githubusercontent.com/Ps1012/telco-customer-churn-analysis/main/WA_Fn-UseC_-Telco-Customer-Churn.csv",
]

if not DATA_PATH.exists():
    downloaded = False
    for url in DATA_URLS:
        try:
            print("Trying:", url)
            urllib.request.urlretrieve(url, DATA_PATH)
            print("Downloaded successfully.")
            downloaded = True
            break
        except Exception as e:
            print("Failed:", type(e).__name__)
    if not downloaded:
        raise RuntimeError(
            "Automatic download failed. Upload the IBM Telco Customer Churn CSV "
            "to Colab and save it as project_3_outputs/data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
        )
else:
    print("Dataset already exists:", DATA_PATH)

raw_df = pd.read_csv(DATA_PATH)

print("\nRaw shape:", raw_df.shape)
display(raw_df.head())

## 3. Data understanding

We inspect columns, types, missing values, duplicates and churn distribution before changing the dataset.

In [ ]:
print("Columns:")
print(raw_df.columns.tolist())

print("\nData types:")
display(raw_df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(raw_df.isna().sum().sort_values(ascending=False).to_frame("missing"))

print("\nExact duplicate rows:", raw_df.duplicated().sum())

print("\nChurn distribution:")
display(raw_df["Churn"].value_counts(dropna=False).to_frame("customers"))

print("\nChurn percentage:")
display((raw_df["Churn"].value_counts(normalize=True) * 100).round(2).to_frame("percent"))

## 4. Data cleaning

Important cleaning steps:
- `TotalCharges` contains blank strings for some new customers, so it must be converted to numeric.
- Customer IDs are checked for duplicates.
- Text values are stripped of accidental spaces.
- Rows missing the target or essential identifier are removed.

In [ ]:
df = raw_df.copy()

# Strip whitespace from object columns.
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

# Convert TotalCharges to numeric. Blank strings become NaN.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Missing TotalCharges after numeric conversion:", df["TotalCharges"].isna().sum())

# For customers with zero tenure, missing TotalCharges is logically treated as 0.
df.loc[(df["tenure"] == 0) & (df["TotalCharges"].isna()), "TotalCharges"] = 0

# Any remaining missing TotalCharges values use the median.
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# Remove exact duplicates, if any.
df = df.drop_duplicates().copy()

# Keep rows with valid customer ID and churn target.
df = df[df["customerID"].notna() & df["Churn"].isin(["Yes", "No"])].copy()

# Binary target for machine learning.
df["ChurnFlag"] = (df["Churn"] == "Yes").astype(int)

print("Clean shape:", df.shape)
print("Duplicate customer IDs:", df["customerID"].duplicated().sum())
print("Remaining missing values:", int(df.isna().sum().sum()))
print("Overall churn rate: {:.2%}".format(df["ChurnFlag"].mean()))

display(df.head())

## 5. Feature engineering

These features make the analysis easier to explain to a business audience.

In [ ]:
# Tenure groups
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, 60, np.inf],
    labels=["0-12 Months", "13-24 Months", "25-48 Months", "49-60 Months", "61+ Months"]
)

# Monthly charge bands
df["MonthlyChargeBand"] = pd.cut(
    df["MonthlyCharges"],
    bins=[-np.inf, 35, 70, 100, np.inf],
    labels=["Low", "Medium", "High", "Very High"]
)

# Number of subscribed/active services
service_columns = [
    "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
]
df["ServiceCount"] = sum((df[col] == "Yes").astype(int) for col in service_columns)

# Contract risk description
contract_risk_map = {
    "Month-to-month": "High Contract Risk",
    "One year": "Medium Contract Risk",
    "Two year": "Low Contract Risk"
}
df["ContractRisk"] = df["Contract"].map(contract_risk_map)

# Estimated customer value proxy
df["EstimatedCustomerValue"] = df["MonthlyCharges"] * df["tenure"]

print("New engineered fields:")
display(
    df[[
        "customerID", "TenureGroup", "MonthlyChargeBand",
        "ServiceCount", "ContractRisk", "EstimatedCustomerValue"
    ]].head()
)

## 6. Exploratory Data Analysis (EDA)

We first examine overall churn and then compare churn rates across important customer characteristics.

In [ ]:
def save_churn_rate_chart(column, filename, title, rotate=0):
    summary = (
        df.groupby(column, observed=False)["ChurnFlag"]
          .agg(["mean", "count"])
          .reset_index()
    )
    summary["ChurnRatePct"] = summary["mean"] * 100
    summary = summary.sort_values("ChurnRatePct", ascending=False)

    plt.figure(figsize=(9, 5))
    plt.bar(summary[column].astype(str), summary["ChurnRatePct"])
    plt.title(title)
    plt.ylabel("Churn Rate (%)")
    plt.xlabel(column)
    plt.xticks(rotation=rotate, ha="right" if rotate else "center")
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / filename, dpi=160, bbox_inches="tight")
    plt.show()

    display(summary[[column, "count", "ChurnRatePct"]].round(2))
    return summary

# Overall churn
churn_counts = df["Churn"].value_counts()
plt.figure(figsize=(6, 4))
plt.bar(churn_counts.index, churn_counts.values)
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Customers")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "churn_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

contract_summary = save_churn_rate_chart(
    "Contract", "churn_by_contract.png", "Churn Rate by Contract Type", 20
)
tenure_summary = save_churn_rate_chart(
    "TenureGroup", "churn_by_tenure_group.png", "Churn Rate by Tenure Group", 25
)
payment_summary = save_churn_rate_chart(
    "PaymentMethod", "churn_by_payment_method.png", "Churn Rate by Payment Method", 30
)
internet_summary = save_churn_rate_chart(
    "InternetService", "churn_by_internet_service.png", "Churn Rate by Internet Service", 15
)
tech_summary = save_churn_rate_chart(
    "TechSupport", "churn_by_tech_support.png", "Churn Rate by Tech Support", 15
)
security_summary = save_churn_rate_chart(
    "OnlineSecurity", "churn_by_online_security.png", "Churn Rate by Online Security", 15
)

### Monthly charges and churn

We compare the distribution of monthly charges for retained and churned customers.

In [ ]:
plt.figure(figsize=(9, 5))
for churn_value, label in [(0, "Retained"), (1, "Churned")]:
    vals = df.loc[df["ChurnFlag"] == churn_value, "MonthlyCharges"]
    plt.hist(vals, bins=25, alpha=0.55, label=label)

plt.title("Monthly Charges Distribution by Churn")
plt.xlabel("Monthly Charges")
plt.ylabel("Customers")
plt.legend()
plt.tight_layout()
plt.savefig(IMAGES_DIR / "monthly_charges_by_churn.png", dpi=160, bbox_inches="tight")
plt.show()

charge_band_summary = (
    df.groupby("MonthlyChargeBand", observed=False)["ChurnFlag"]
      .agg(["mean", "count"])
      .reset_index()
)
charge_band_summary["ChurnRatePct"] = charge_band_summary["mean"] * 100
display(charge_band_summary[["MonthlyChargeBand", "count", "ChurnRatePct"]].round(2))

## 7. Prepare data for machine learning

`customerID`, the original text target `Churn`, and the numeric target `ChurnFlag` are not used as predictors.

The preprocessing pipeline:
- imputes numeric values
- scales numeric features
- imputes categorical values
- one-hot encodes categorical features

In [ ]:
TARGET = "ChurnFlag"

drop_for_model = [
    "customerID",
    "Churn",
    TARGET,
    "ContractRisk"  # derived directly from Contract; avoid redundant encoding
]

X = df.drop(columns=drop_for_model)
y = df[TARGET]

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()

print("Numeric features:", numeric_features)
print("\nCategorical features:", categorical_features)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training churn rate: {:.2%}".format(y_train.mean()))
print("Test churn rate: {:.2%}".format(y_test.mean()))

## 8. Train multiple classification models

We compare:
1. Logistic Regression — interpretable baseline
2. Decision Tree
3. Random Forest
4. Gradient Boosting

The best model is selected by **ROC-AUC**, while Recall and F1 are also reviewed because missing a likely churner can be expensive.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=2,
        random_state=42
    )
}

trained_models = {}
results = []

for name, estimator in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_prob)
    }

    results.append(metrics)
    trained_models[name] = pipe

results_df = pd.DataFrame(results).sort_values("ROC_AUC", ascending=False).reset_index(drop=True)

print("Model comparison:")
display(results_df.style.format({
    "Accuracy": "{:.3f}",
    "Precision": "{:.3f}",
    "Recall": "{:.3f}",
    "F1": "{:.3f}",
    "ROC_AUC": "{:.3f}"
}))

results_df.to_csv(REPORTS_DIR / "model_comparison.csv", index=False)

## 9. Model comparison chart and best-model selection

In [ ]:
plot_df = results_df.set_index("Model")[["Recall", "F1", "ROC_AUC"]]

plot_df.plot(kind="bar", figsize=(10, 5))
plt.title("Model Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "model_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

BEST_MODEL_NAME = results_df.loc[0, "Model"]
best_model = trained_models[BEST_MODEL_NAME]

print("Selected best model by ROC-AUC:", BEST_MODEL_NAME)

## 10. Evaluate the selected model

The confusion matrix shows the types of classification errors.  
The ROC curve shows the trade-off between true-positive and false-positive rates across probability thresholds.

In [ ]:
best_pred = best_model.predict(X_test)
best_prob = best_model.predict_proba(X_test)[:, 1]

print("Classification report for:", BEST_MODEL_NAME)
print(classification_report(y_test, best_pred, target_names=["Retained", "Churned"]))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, best_pred,
    display_labels=["Retained", "Churned"],
    cmap="Blues",
    ax=ax
)
ax.set_title(f"Confusion Matrix — {BEST_MODEL_NAME}")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test, best_prob, ax=ax)
ax.set_title(f"ROC Curve — {BEST_MODEL_NAME}")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "roc_curve.png", dpi=160, bbox_inches="tight")
plt.show()

print("ROC-AUC: {:.3f}".format(roc_auc_score(y_test, best_prob)))

## 11. Feature importance using permutation importance

Permutation importance measures how much predictive performance falls when a feature is randomly shuffled.  
This works consistently across different model types and gives importance at the original business-feature level.

In [ ]:
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": perm.importances_mean
}).sort_values("Importance", ascending=False)

display(importance_df.head(15))

top_imp = importance_df.head(15).sort_values("Importance")

plt.figure(figsize=(9, 6))
plt.barh(top_imp["Feature"], top_imp["Importance"])
plt.title(f"Top Predictive Features — {BEST_MODEL_NAME}")
plt.xlabel("Permutation Importance (ROC-AUC decrease)")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "feature_importance.png", dpi=160, bbox_inches="tight")
plt.show()

importance_df.to_csv(REPORTS_DIR / "feature_importance.csv", index=False)

## 12. Refit the selected model on all cleaned data

After evaluating the model on the held-out test set, we refit the selected pipeline using all cleaned customer records so it can generate final portfolio risk scores.

In [ ]:
final_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", models[BEST_MODEL_NAME])
])

final_model.fit(X, y)

MODEL_PATH = MODELS_DIR / "best_churn_model.pkl"
joblib.dump(final_model, MODEL_PATH)

print("Saved final model:", MODEL_PATH)

## 13. Generate churn probabilities and risk bands

Risk bands:
- **Low Risk:** probability < 30%
- **Medium Risk:** 30% to < 60%
- **High Risk:** 60% or above

These thresholds are business-friendly starting points and can be adjusted later based on campaign capacity and retention cost.

In [ ]:
all_prob = final_model.predict_proba(X)[:, 1]
all_pred = (all_prob >= 0.50).astype(int)

scored_df = df.copy()
scored_df["ChurnProbability"] = all_prob
scored_df["PredictedChurn"] = np.where(all_pred == 1, "Yes", "No")

scored_df["RiskBand"] = pd.cut(
    scored_df["ChurnProbability"],
    bins=[-0.001, 0.30, 0.60, 1.0],
    labels=["Low Risk", "Medium Risk", "High Risk"],
    include_lowest=True,
    right=False
)

# Monthly recurring revenue exposure proxy.
scored_df["MonthlyChargesAtRisk"] = np.where(
    scored_df["RiskBand"].astype(str) == "High Risk",
    scored_df["MonthlyCharges"],
    0
)

risk_summary = (
    scored_df.groupby("RiskBand", observed=False)
             .agg(
                 Customers=("customerID", "count"),
                 AvgChurnProbability=("ChurnProbability", "mean"),
                 MonthlyCharges=("MonthlyCharges", "sum"),
                 ActualChurnRate=("ChurnFlag", "mean")
             )
             .reset_index()
)

risk_summary["AvgChurnProbabilityPct"] = risk_summary["AvgChurnProbability"] * 100
risk_summary["ActualChurnRatePct"] = risk_summary["ActualChurnRate"] * 100

display(
    risk_summary[
        ["RiskBand", "Customers", "AvgChurnProbabilityPct", "MonthlyCharges", "ActualChurnRatePct"]
    ].round(2)
)

plt.figure(figsize=(7, 4))
risk_counts = scored_df["RiskBand"].value_counts().reindex(["Low Risk", "Medium Risk", "High Risk"])
plt.bar(risk_counts.index.astype(str), risk_counts.values)
plt.title("Customers by Churn Risk Band")
plt.ylabel("Customers")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "risk_band_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

## 14. Export cleaned data, churn predictions and Power BI-ready data

In [ ]:
cleaned_path = DATA_DIR / "telco_churn_cleaned.csv"
pred_path = DATA_DIR / "churn_risk_predictions.csv"
powerbi_path = POWERBI_DIR / "churn_powerbi.csv"

df.to_csv(cleaned_path, index=False)

prediction_columns = [
    "customerID",
    "Churn",
    "ChurnFlag",
    "ChurnProbability",
    "PredictedChurn",
    "RiskBand",
    "MonthlyCharges",
    "MonthlyChargesAtRisk",
    "tenure",
    "Contract",
    "PaymentMethod",
    "InternetService",
    "TechSupport",
    "OnlineSecurity"
]
scored_df[prediction_columns].to_csv(pred_path, index=False)

# Power BI dataset contains the cleaned customer data plus model outputs.
powerbi_df = scored_df.copy()

# Convert categorical pandas dtypes to plain strings for easier Power BI import.
for col in powerbi_df.select_dtypes(include="category").columns:
    powerbi_df[col] = powerbi_df[col].astype(str)

powerbi_df.to_csv(powerbi_path, index=False)

print("Saved:")
print("-", cleaned_path)
print("-", pred_path)
print("-", powerbi_path)
print("\nPower BI rows:", len(powerbi_df))
display(powerbi_df.head())

## 15. Business findings and retention recommendations

This section produces data-grounded findings from the current dataset.

In [ ]:
overall_churn = scored_df["ChurnFlag"].mean()
high_risk_customers = int((scored_df["RiskBand"].astype(str) == "High Risk").sum())
high_risk_monthly = scored_df.loc[
    scored_df["RiskBand"].astype(str) == "High Risk", "MonthlyCharges"
].sum()

top_contract = (
    scored_df.groupby("Contract")["ChurnFlag"].mean().sort_values(ascending=False).index[0]
)
top_payment = (
    scored_df.groupby("PaymentMethod")["ChurnFlag"].mean().sort_values(ascending=False).index[0]
)
top_internet = (
    scored_df.groupby("InternetService")["ChurnFlag"].mean().sort_values(ascending=False).index[0]
)
top_tenure = (
    scored_df.groupby("TenureGroup", observed=False)["ChurnFlag"].mean().sort_values(ascending=False).index[0]
)

findings = f"""
CUSTOMER CHURN & RETENTION ANALYTICS — BUSINESS FINDINGS
========================================================

Dataset size: {len(scored_df):,} customers
Overall churn rate: {overall_churn:.2%}
Selected model: {BEST_MODEL_NAME}
Test ROC-AUC: {results_df.loc[0, 'ROC_AUC']:.3f}
High-risk customers: {high_risk_customers:,}
Monthly charges represented by high-risk customers: ${high_risk_monthly:,.2f}

Key observed churn patterns
---------------------------
Highest-churn contract type: {top_contract}
Highest-churn payment method: {top_payment}
Highest-churn internet service group: {top_internet}
Highest-churn tenure group: {top_tenure}

Retention recommendations
-------------------------
1. Prioritise high-risk customers for proactive retention outreach.
2. Review incentives that encourage movement away from the highest-churn contract type ({top_contract}).
3. Investigate the customer experience associated with the highest-churn payment method ({top_payment}).
4. Review service quality, pricing and support for the highest-churn internet-service group ({top_internet}).
5. Create onboarding and early-life retention campaigns for the highest-churn tenure group ({top_tenure}).
6. Offer targeted Tech Support / Online Security bundles where churn risk is elevated.
7. Use churn probability and customer value together when prioritising retention spend.

Model note
----------
The best model was selected by ROC-AUC on a held-out test set. Probability thresholds and
risk-band cutoffs should be tuned to business campaign capacity, intervention cost and the
financial value of retaining a customer.
"""

print(findings)

(REPORTS_DIR / "business_findings.txt").write_text(findings, encoding="utf-8")

## 16. Save a reusable Python pipeline script

In [ ]:
script = r"""# Customer Churn Prediction Pipeline
# Generated by Hassan Project 3 notebook.

from pathlib import Path
import pandas as pd
import numpy as np
import joblib

def load_model(model_path="models/best_churn_model.pkl"):
    return joblib.load(model_path)

def score_customers(model, feature_df):
    probability = model.predict_proba(feature_df)[:, 1]
    prediction = (probability >= 0.50).astype(int)

    result = feature_df.copy()
    result["ChurnProbability"] = probability
    result["PredictedChurn"] = np.where(prediction == 1, "Yes", "No")
    result["RiskBand"] = pd.cut(
        result["ChurnProbability"],
        bins=[-0.001, 0.30, 0.60, 1.0],
        labels=["Low Risk", "Medium Risk", "High Risk"],
        include_lowest=True,
        right=False
    )
    return result
"""

script_path = SRC_DIR / "churn_pipeline.py"
script_path.write_text(script, encoding="utf-8")
print("Saved:", script_path)

## 17. Power BI dashboard plan

Use `project_3_outputs/powerbi/churn_powerbi.csv` in Power BI Desktop.

### Page 1 — Executive Overview
- Total Customers
- Churned Customers
- Churn Rate
- Retained Customers
- High-Risk Customers
- Monthly Charges at Risk

### Page 2 — Churn Drivers
- Churn by Contract
- Churn by Tenure Group
- Churn by Payment Method
- Churn by Internet Service
- Churn by Tech Support
- Churn by Monthly Charge Band

### Page 3 — Customer Segments
- Risk Band
- Service Count
- Customer Value
- Tenure
- Monthly Charges
- Customer segments and profiles

### Page 4 — Retention & High-Risk Customers
- High-risk customer table
- Churn Probability
- Monthly Charges at Risk
- Contract
- Payment Method
- Tech Support
- Online Security
- Retention prioritisation filters

## 18. Create requirements file and ZIP all outputs

In [ ]:
requirements = '''pandas
numpy
matplotlib
scikit-learn
joblib
'''

(PROJECT_DIR / "requirements.txt").write_text(requirements, encoding="utf-8")

# Save a concise README for generated outputs.
generated_readme = f'''# Customer Churn Prediction & Retention Analytics

Generated by `Hassan_Project_3_Customer_Churn_ML_PowerBI_READY.ipynb`.

## Selected model
{BEST_MODEL_NAME}

## Test ROC-AUC
{results_df.loc[0, "ROC_AUC"]:.3f}

## Outputs
- cleaned customer data
- customer churn-risk predictions
- Power BI-ready CSV
- trained model
- model comparison
- confusion matrix
- ROC curve
- permutation feature importance
- EDA charts
- business findings
- reusable Python scoring script
'''

(PROJECT_DIR / "README_GENERATED.md").write_text(generated_readme, encoding="utf-8")

import shutil

zip_name = "Hassan_Project_3_Colab_Outputs"
zip_path = shutil.make_archive(zip_name, "zip", PROJECT_DIR)

print("Created ZIP:", zip_path)
print("\nProject 3 analysis is complete.")
print("Download the ZIP from the Colab Files panel after running all cells.")

# Finished

If every cell ran successfully, the analysis and machine-learning stage of Project 3 is complete.

### Main generated files
- `project_3_outputs/data/telco_churn_cleaned.csv`
- `project_3_outputs/data/churn_risk_predictions.csv`
- `project_3_outputs/powerbi/churn_powerbi.csv`
- `project_3_outputs/models/best_churn_model.pkl`
- `project_3_outputs/src/churn_pipeline.py`
- `project_3_outputs/reports/model_comparison.csv`
- `project_3_outputs/reports/feature_importance.csv`
- `project_3_outputs/reports/business_findings.txt`
- charts in `project_3_outputs/images/`
- `Hassan_Project_3_Colab_Outputs.zip`

The next stage is the **Power BI dashboard** and GitHub packaging.